# Setup Check
**Run this notebook before the workshop starts.**

Each cell prints `✓ PASS` or `✗ FAIL` with a fix hint.
You only need the cells marked *(required)* to participate in all exercises.
Docker / OpenSearch is *optional* — a local in memory fallback is provided.

There are known package dependency concerns. 

In recent years, the new Mac Silicon architecture, changes to setuptools and
pip, numpy v2.0, and changes to how Huggingface libraries are managed mean that
some of the latest versions of libraries are incompatible with older operating
systems. In order to meet the needs of attendees who may be participating on
different operating systems, I have made some adjustments to the required package
versions, often choosing to use an older version of a library, rather than the
latest version.

Friends have tested the versions specified in `pyproject.toml` to ensure that they work together on Windows, Mac Intel, Mac Silicon and Linux platforms.

However, it is possible that you may encounter package conflicts. Please contact me at least 48 hours in advance of the workshop, and I will do my best to find a solution before the workshop. I will also be available during the workshop to help troubleshoot any issues that arise, but may have limited time to help.

# Installation Instructions
Review the README.md and run installation commands in your terminal. When you are done, run this
notebook to check that all is working.  

**Run these commands in your terminal from the repo root:**
```
uv sync    
docker compose up -d
```

In [ ]:
# ── Project virtual environment (required) ────────────────────────────────
# Every check below runs against THIS interpreter. If it isn't the .venv
# created by `uv sync`, other checks may green-light the wrong environment.
import sys
from pathlib import Path

in_venv = sys.prefix != sys.base_prefix
venv_name = Path(sys.prefix).name if in_venv else "(system)"

print(f"   executable : {sys.executable}")
print(f"   sys.prefix : {sys.prefix}")
print(f"   working dir: {Path.cwd()}")

if in_venv and venv_name == ".venv":
    print(f"✓ PASS  running inside a project venv ({venv_name})")
else:
    if in_venv:
        print(f"⚠ WARN  running inside venv '{venv_name}', expected '.venv'")
    else:
        print("⚠ WARN  not running inside a virtual environment")
    print("        If later checks fail, re-launch Jupyter with:")
    print("        uv run jupyter lab notebooks/00_setup_check.ipynb")


In [ ]:
# ── Python version (required) ──────────────────────────────────────────────
# pyproject.toml requires >=3.12,<3.13. `uv sync` will install a matching
# Python automatically; this cell verifies the running kernel actually got it.
import sys

v = sys.version_info

if v.major == 3 and v.minor == 12:
    print(f"✓ PASS  Python {v.major}.{v.minor}.{v.micro}")
else:
    print(f"✗ FAIL  Python {v.major}.{v.minor}.{v.micro} — project requires 3.12.x")
    print("        Fix:  uv python install 3.12 && uv sync")
    print("        Then re-launch Jupyter from the project root:")
    print("        uv run jupyter lab notebooks/00_setup_check.ipynb")
    print("        (Fallback if you don't use uv: pyenv install 3.12 or python.org installer)")


In [ ]:
# ── uv package manager (required) ─────────────────────────────────────────
import shutil

if shutil.which("uv") is None:
    print("✗ FAIL  uv is not installed.")
    print("        Install: https://docs.astral.sh/uv/getting-started/installation/")
else:
    print("✓ PASS  uv is installed")
    print("        Next step: run 'uv sync' from the repo root")

In [ ]:
# ── Core packages (required) ───────────────────────────────────────────────
key_packages = [
    ("sentence_transformers", "sentence-transformers"),
    ("bertopic", "bertopic"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("emoji", "emoji"),
]
all_ok = True
for module, pkg in key_packages:
    try:
        __import__(module)
        print(f"✓ PASS  {pkg}")
    except ImportError:
        print(f"✗ FAIL  {pkg}  →  run: `uv sync` or `uv add {pkg}`")
        all_ok = False
if all_ok:
    print("\nAll key packages installed!")

In [ ]:
# ── Files (required) ────────────────────────────────────────────────
from pathlib import Path

# Detect repo root by looking for a known marker file (sample_posts.json is
# unique to this repo, so it's a reliable anchor).
if Path("../sample_posts.json").exists():
    repo_root = Path("..")
elif Path("./sample_posts.json").exists():
    repo_root = Path(".")
else:
    raise RuntimeError(
        f"Could not locate repo root from cwd: {Path.cwd()}. "
        "Launch with: uv run jupyter lab notebooks/00_setup_check.ipynb"
    )

# Create output directory if it doesn't exist
output_dir = repo_root / "output"
output_dir.mkdir(exist_ok=True)
print(f"✓ PASS  output/  ({'already existed' if output_dir.exists() else 'created'})")

required_files = [
    "sample_posts.json",
]
for f in required_files:
    p = repo_root / f
    if p.exists():
        print(f"✓ PASS  {f}")
    else:
        print(f"✗ FAIL  {f}  →  make sure you cloned the full repo")


In [ ]:
# ── Embedding model (required — downloads ~90 MB on first run) ─────────────
import time

from sentence_transformers import SentenceTransformer

print("Loading all-MiniLM-L6-v2 (~90 MB on first run, <1s after — cached at ~/.cache/huggingface)...")

t0 = time.perf_counter()
try:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    emb = model.encode("hello world")
    assert emb.shape == (384,)
    elapsed = time.perf_counter() - t0
    print(f"✓ PASS  all-MiniLM-L6-v2  (dim={emb.shape[0]}, loaded in {elapsed:.1f}s)")
except Exception as e:
    elapsed = time.perf_counter() - t0
    print(f"✗ FAIL  ({elapsed:.1f}s)  {type(e).__name__}: {e}")
    print("        If this is a network error, retry on better Wi-Fi.")
    print("        If it's an ETag/cache error, clear the cache:")
    print("          rm -rf ~/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2")


In [ ]:
# ── Docker (optional — runs Elasticsearch and Ollama) ─────────────────────
# Four states: CLI missing (SKIP), daemon down (WARN),
# `docker compose` v2 missing (WARN), all good (PASS).
import shutil
import subprocess

if shutil.which("docker") is None:
    print("○ SKIP  Docker not installed — in-memory fallback will be used.")
    print("        Install Docker Desktop: https://www.docker.com/products/docker-desktop/")
else:
    info = subprocess.run(
        ["docker", "info", "--format", "{{.ServerVersion}}"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    if info.returncode != 0:
        print("⚠ WARN  Docker installed but the daemon is not running.")
        print("        Start Docker Desktop (or `sudo systemctl start docker` on Linux),")
        print("        then run: docker compose up -d")
    else:
        server_version = info.stdout.strip()
        compose = subprocess.run(
            ["docker", "compose", "version", "--short"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        if compose.returncode != 0:
            print(f"⚠ WARN  Docker {server_version} running, but `docker compose` (v2) not available.")
            print("        Workshop docs assume compose v2 (one binary, subcommand syntax).")
            print("        Upgrade Docker Desktop, or install the compose plugin.")
        else:
            print(f"✓ PASS  Docker {server_version}, compose {compose.stdout.strip()}")


In [ ]:
# ── Elasticsearch (optional — Docker install) ────────────────────────────
# Hits /_cluster/health and asks ES to block up to 30s for the cluster to
# reach `yellow`, so this call also acts as a startup-completion barrier.
import httpx

ES_URL = "http://localhost:9201"
HEALTH = f"{ES_URL}/_cluster/health?wait_for_status=yellow&timeout=30s"

try:
    r = httpx.get(HEALTH, timeout=35.0)
    r.raise_for_status()
    h = r.json()
    status = h.get("status")
    if status in ("yellow", "green"):
        print(
            f"✓ PASS  Elasticsearch {status}  "
            f"(nodes={h.get('number_of_nodes')}, active_shards={h.get('active_shards')})"
        )
    else:
        print(f"⚠ WARN  Elasticsearch responding but cluster status is {status!r}")
        print("        Check: docker compose logs elasticsearch")
except httpx.ConnectError:
    print("○ SKIP  Elasticsearch not running — in-memory fallback will be used.")
    print("        To enable: docker compose up -d  (from repo root)")
except httpx.ReadTimeout:
    print("⚠ WARN  Elasticsearch is starting but didn't reach yellow within 30s.")
    print("        Wait a moment and re-run this cell. If it persists:")
    print("          docker compose logs elasticsearch")
except httpx.HTTPStatusError as e:
    print(f"⚠ WARN  Elasticsearch returned HTTP {e.response.status_code}")
    print("        Check: docker compose logs elasticsearch")
except Exception as e:
    print(f"⚠ WARN  Unexpected error contacting Elasticsearch: {type(e).__name__}: {e}")


In [ ]:
# ── Ollama (optional — local LLM for topic labels) ────────────────────────
# Verifies Ollama is reachable AND the expected model has been pulled.
# Without Ollama, topic labels fall back to KeyBERT keyword phrases.
import os

import httpx

OLLAMA_URL = "http://localhost:11434"
EXPECTED_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:3b")

try:
    r = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5.0)
    r.raise_for_status()
    models = [m.get("name") for m in r.json().get("models", [])]
    if EXPECTED_MODEL in models:
        print(f"✓ PASS  Ollama running, {EXPECTED_MODEL} pulled  ({len(models)} model(s) total)")
    else:
        print(f"⚠ WARN  Ollama running but {EXPECTED_MODEL!r} is not pulled.")
        print(f"        Pull it:  ollama pull {EXPECTED_MODEL}")
        if models:
            print(f"        Currently pulled: {', '.join(models)}")
except (httpx.ConnectError, httpx.ConnectTimeout):
    print("○ SKIP  Ollama not reachable — topic labels will fall back to KeyBERT phrases.")
    print("        To enable: docker compose up -d  (or `ollama serve` for native install)")
except httpx.ReadTimeout:
    print("⚠ WARN  Ollama accepted the connection but didn't respond within 5s.")
    print("        Check `ollama serve` logs, or restart Ollama and retry.")
except httpx.HTTPStatusError as e:
    print(f"⚠ WARN  Ollama returned HTTP {e.response.status_code}")
except Exception as e:
    print(f"⚠ WARN  Unexpected error contacting Ollama: {type(e).__name__}: {e}")


In [ ]:
# ── Streamlit (app.py demo) ─────────────────────────
try:
    import streamlit

    print(f"✓ PASS  streamlit {streamlit.__version__}")
except ImportError:
    print("○ FAIL  streamlit not installed — run: uv sync (from project root")

## Run the test suite

Run the non-exercise tests to confirm your environment is wired up correctly. All tests should pass before you begin — the `exercise`-marked tests are intentionally stubbed out and will fail until you complete them. Run the cell below, or from terminal while at this project root, run:

```bash
uv run pytest -m "not exercise"
```

In [ ]:
!cd .. && uv run pytest -m "not exercise"